In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="transformers",
    notebook_name="04_transformer_block.ipynb"
)

# The Complete Transformer Block

## The Mystery Worth Solving

You have built attention. You understand how words look at each other. But here is the thing: attention alone is not enough. If you stack 96 attention layers directly, signals grow out of control, gradients vanish, and training collapses.

The transformer block is the engineering solution to that problem. It wraps attention with three stabilizing mechanisms -- layer normalization, residual connections, and a feed-forward network -- that together allow you to stack 96 layers without everything falling apart.

By the end of this notebook, you will have assembled a complete transformer block from the ground up, and you will understand *why* each piece is there.

---

In previous notebooks, we built the individual pieces. Now we'll assemble them into a complete **transformer block** — the repeating unit that makes transformers work.

In this notebook, we'll:

1. Implement **layer normalization** from scratch
2. Understand **residual connections** (skip connections)
3. Build the **feed-forward network** (FFN)
4. Assemble a complete **transformer block**
5. **Stack multiple blocks** to create a mini transformer

**Prerequisites:** Complete notebooks [01](./01_attention_mechanisms.ipynb), [02](./02_multi_head_attention.ipynb), and [03](./03_positional_encoding.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 12

np.random.seed(42)
print("Setup complete!")

## Part 1: Reusable Components from Previous Notebooks

In [ ]:
# === Reusable functions from previous notebooks ===

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)
    weights = softmax(scores)
    return weights @ V, weights


def sinusoidal_positional_encoding(max_len, d_model):
    PE = np.zeros((max_len, d_model))
    for pos in range(max_len):
        for i in range(0, d_model, 2):
            denom = 10000 ** (i / d_model)
            PE[pos, i] = np.sin(pos / denom)
            if i + 1 < d_model:
                PE[pos, i + 1] = np.cos(pos / denom)
    return PE


print("Building blocks loaded!")

## Part 2: Layer Normalization

### Layer 1: The Grading-on-a-Curve Analogy

Imagine you're a teacher with two test scores: one student scored 95/100, another scored 30/100. To compare them fairly with students from a different school where tests are scored 0–10, you standardize everyone to the same scale: mean = 0, standard deviation = 1.

That's layer normalization. Before sending a vector of numbers to the next layer, you rescale it so:
- Mean ≈ 0
- Standard deviation ≈ 1

This prevents numbers from "snowballing" (getting larger and larger) as they pass through dozens of layers.

**What this analogy gets right:** Standardizing scores to mean=0, std=1 makes them comparable across different tests -- the same way LayerNorm makes activations comparable across different layers, preventing values from snowballing out of control.

**Where this analogy breaks down:** A teacher standardizes grades to be fair to students (batch dimension). LayerNorm standardizes within a single word's feature vector (feature dimension). This is exactly why LayerNorm handles variable-length sequences better than BatchNorm -- it never mixes real tokens with padding tokens from other sequences in the batch.

### Layer 2: The Full Formula

```
LayerNorm(x) = γ × (x - μ) / (σ + ε) + β
```

Where:
- **x** = input vector (one word's representation, shape: d_model)
- **μ = mean(x)** = average of all d_model values in this vector
- **σ = std(x)** = standard deviation of all d_model values
- **ε** = small constant (e.g., 1e-6) to prevent division by zero
- **γ** = learned scale parameter (initialized to 1), shape: d_model
- **β** = learned shift parameter (initialized to 0), shape: d_model

The (x - μ) / σ part normalizes x to zero mean and unit variance. Then γ and β allow the network to learn a different scale and shift if that's more useful — so if the task benefits from larger or shifted activations, the model can learn that.

**Why LayerNorm and not BatchNorm?**

BatchNorm normalizes across the batch dimension (average across all sequences in the batch). LayerNorm normalizes across the feature dimension (average across all d_model values for one word).

Two reasons LayerNorm wins for transformers:
1. Sequence lengths vary. BatchNorm at position 100 would average over sequences that might not even have 100 tokens — you'd be normalizing real tokens with padding.
2. Train/test consistency. BatchNorm's statistics depend on batch size and composition. LayerNorm's statistics depend only on the single input vector — same behavior at training and inference, regardless of batch size.

**Parameter count**: 2 × d_model per LayerNorm (γ and β). For d_model = 768: 1,536 parameters. Tiny compared to attention (2.36M) — LayerNorm adds almost nothing.

In [ ]:
class LayerNorm:
    """Layer Normalization — normalizes each vector independently."""
    
    def __init__(self, d_model, eps=1e-6):
        self.gamma = np.ones(d_model)    # Learnable scale (initialized to 1)
        self.beta = np.zeros(d_model)    # Learnable shift (initialized to 0)
        self.eps = eps                    # Small number to prevent division by zero
    
    def __call__(self, x):
        # Compute mean and std for each word vector
        mean = x.mean(axis=-1, keepdims=True)
        std = x.std(axis=-1, keepdims=True)
        
        # Normalize
        x_norm = (x - mean) / (std + self.eps)
        
        # Scale and shift (learnable parameters)
        return self.gamma * x_norm + self.beta


# Demonstrate layer normalization
d_model = 8
ln = LayerNorm(d_model)

# Before normalization: numbers are all over the place
x = np.array([
    [150.0, -200.0, 3.0, 50.0, 80.0, -10.0, 25.0, 100.0],
    [0.001, 0.002, 0.003, 0.001, 0.002, 0.001, 0.003, 0.002],
])

x_normed = ln(x)

print("Before LayerNorm:")
print(f"  Word 1: [{', '.join(f'{v:>8.1f}' for v in x[0])}]")
print(f"    mean = {x[0].mean():.1f}, std = {x[0].std():.1f}")
print(f"  Word 2: [{', '.join(f'{v:>8.4f}' for v in x[1])}]")
print(f"    mean = {x[1].mean():.4f}, std = {x[1].std():.4f}")
print()
print("After LayerNorm:")
print(f"  Word 1: [{', '.join(f'{v:>8.3f}' for v in x_normed[0])}]")
print(f"    mean = {x_normed[0].mean():.6f}, std = {x_normed[0].std():.3f}")
print(f"  Word 2: [{', '.join(f'{v:>8.3f}' for v in x_normed[1])}]")
print(f"    mean = {x_normed[1].mean():.6f}, std = {x_normed[1].std():.3f}")
print()
print("Both vectors now have mean ≈ 0 and std ≈ 1, regardless of original scale!")

In [ ]:
# Visualize the effect of layer normalization
np.random.seed(42)
x_demo = np.random.randn(10, 32)

# Simulate what happens after many layers (numbers grow)
for _ in range(5):
    W = np.random.randn(32, 32) * 0.5
    x_demo = x_demo @ W

x_demo_normed = LayerNorm(32)(x_demo)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].boxplot([x_demo[i] for i in range(10)])
axes[0].set_title('Before LayerNorm\n(values exploded after many layers)', fontsize=13)
axes[0].set_xlabel('Word position', fontsize=11)
axes[0].set_ylabel('Value', fontsize=11)

axes[1].boxplot([x_demo_normed[i] for i in range(10)])
axes[1].set_title('After LayerNorm\n(stable, centered values)', fontsize=13)
axes[1].set_xlabel('Word position', fontsize=11)
axes[1].set_ylabel('Value', fontsize=11)

plt.tight_layout()
plt.show()

print("Without LayerNorm, values grow out of control.")
print("With LayerNorm, values stay in a stable range — training is much easier!")

## Part 3: Residual Connections (Skip Connections)

### Layer 1: The Recipe Safety Net

Imagine you're following a complex recipe with 20 steps. Each step says "transform the dish somehow." By step 15, if something went wrong at step 7, you've lost the original dish completely.

A residual connection is like keeping a photo of your dish before EACH step. If step 7 made things worse, you can still see the original dish and add it back in. The network learns to make SMALL IMPROVEMENTS rather than COMPLETE TRANSFORMATIONS.

**What this analogy gets right:** Instead of learning a complete transformation from scratch, each layer only needs to learn the *difference* -- the improvement over what already exists. That is much easier to learn, and it is exactly why residual networks train successfully at great depth.

**Where this analogy breaks down:** In cooking, if a step makes things worse, you discard it and revert to the photo. In residual connections, the layer output is always *added* to the input -- you cannot selectively discard a layer's contribution. The network instead learns to output near-zero values when a layer is not helpful, which is the mathematical equivalent of skipping that layer.

Formula:
```
output = sublayer(x) + x    ← the "+ x" is the residual
```

### Layer 2: Why This Solves the Vanishing Gradient Problem

The gradient of the loss with respect to the layer input x is:

```
∂L/∂x = ∂L/∂output × ∂output/∂x
       = ∂L/∂output × (∂sublayer(x)/∂x + 1)
```

The "+1" term means the gradient is NEVER smaller than ∂L/∂output × 1 — even if sublayer(x) has vanishing gradients (∂sublayer/∂x ≈ 0), the gradient still flows back as a constant 1× scaling.

For N stacked residual blocks, the gradient path from the output to the input has 2^N possible paths (go through each sublayer or skip it). The gradient is the sum over all these paths. Even if all sublayers have vanishing gradients, the all-skipping path (1 × 1 × ... × 1 = 1) ensures signal flows.

This is why transformers with 96+ layers can train: at initialization, the gradient flows primarily through the "all-skip" path (because sublayers output near-zero at random init), giving a well-conditioned gradient. As training progresses, individual layers learn to contribute, and the skip paths prevent any single layer from blocking gradient flow.

**Pre-LN vs Post-LN:**

```
Post-LN (original paper):      Pre-LN (modern, default):
x → sublayer → + residual      x → LayerNorm → sublayer → + residual
    → LayerNorm                
```

Pre-LN (used by GPT-2 through LLaMA) is more stable because the residual stream x is never modified by normalization — gradients flow through the residual path unimpeded. Post-LN applies normalization AFTER the residual addition, which can cause large gradient magnitudes early in training (requiring careful LR warmup).

In [ ]:
# Demonstrate residual connections

def simulate_layers(x, n_layers, use_residual=False):
    """Simulate passing through multiple layers."""
    d = x.shape[-1]
    history = [np.linalg.norm(x)]
    
    for i in range(n_layers):
        # Simulate a layer (random transformation)
        W = np.random.randn(d, d) * 0.5
        layer_output = np.tanh(x @ W)
        
        if use_residual:
            x = layer_output + x  # Residual: add input back!
        else:
            x = layer_output       # No residual: input is lost
        
        history.append(np.linalg.norm(x))
    
    return x, history


# Compare with and without residual connections
np.random.seed(42)
x_input = np.random.randn(5, 32)

np.random.seed(42)
_, history_no_res = simulate_layers(x_input.copy(), 20, use_residual=False)

np.random.seed(42)
_, history_with_res = simulate_layers(x_input.copy(), 20, use_residual=True)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(history_no_res, 'r-o', label='Without Residual', markersize=4, linewidth=2)
ax.plot(history_with_res, 'b-o', label='With Residual', markersize=4, linewidth=2)
ax.set_xlabel('Layer Number', fontsize=13)
ax.set_ylabel('Signal Magnitude (norm)', fontsize=13)
ax.set_title('Effect of Residual Connections Through 20 Layers', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Without residuals: signal can shrink (vanish) or explode")
print("With residuals: signal stays stable because original input is always preserved")
print("\nThis is why deep transformers (96+ layers) can work!")

## Part 4: Feed-Forward Network (FFN)

### Layer 1: Private Thinking Time

After attention lets all words talk to each other (sharing information), each word gets "private thinking time" through the FFN. This is where each word processes the information it just gathered.

Crucially: words are processed **independently** here. "cat" goes through the FFN by itself; "sat" goes through separately. No interaction between words. This is the opposite of attention, which is all about interaction.

The FFN expands the vector to a much larger size (typically 4×), does some nonlinear computation, then compresses back:

```
word_vector (512) → expand to 2048 → activate → compress to 512
```

The expansion gives the model more "space to think" — more dimensions = more independent features to work with.

**What this analogy gets right:** After a group discussion (attention), you need private processing time to integrate what you heard. The FFN provides exactly that -- each word processes the gathered context independently, without interference from other words.

**Where this analogy breaks down:** Human private thinking can generate qualitatively new insights not present in the discussion. The FFN is a fixed mathematical function (two linear layers with a nonlinearity) -- it can only transform and recombine what attention gathered, not generate new information from outside the input. Also, human thinking is partly creative and stochastic. The FFN is deterministic: the same input always produces the same output.

### Layer 2: The FFN as a Key-Value Memory

Geva et al. (2021) showed that FFN layers function as **key-value memories**. Each row of W₁ is a "key" that patterns in the input activate. Each column of W₂ is a "value" that gets contributed to the output when its key is activated.

Formally: FFN(x) = W₂ × ReLU(W₁ × x + b₁) + b₂

The i-th neuron in the hidden layer fires (via ReLU) when W₁[i] · x > 0. W₁[i] is a key that matches certain input patterns (e.g., "verbs in past tense"). When it fires, W₂[:, i] contributes its corresponding value to the output.

In this view, d_ff (the hidden dimension) is the number of "key-value pairs" stored per token per layer. Larger d_ff = more knowledge can be stored. This is why d_ff ≈ 65% of transformer parameters despite looking like a simple two-layer network.

**GELU vs ReLU:**

Modern transformers (GPT-2 onward) use GELU instead of ReLU. ReLU is a hard cutoff: negative → 0, positive → identity. GELU is smooth: it "softly" passes small positive values and suppresses small negative values. GELU has slightly better empirical performance and smoother gradient flow, at near-identical compute cost.

SwiGLU (used in LLaMA): FFN(x) = (W₁x × gate) × W₃x where gate = sigmoid(W₂x). Uses 3 matrices instead of 2, effectively implementing a learned gating mechanism. With d_ff ≈ 2.67 × d_model (instead of 4×), SwiGLU matches or exceeds ReLU/GELU FFN quality with similar parameter count.

**Parameter count per FFN**: 2 × d_model × d_ff + d_ff + d_model (biases are small). For d_model=768, d_ff=3072: 2 × 768 × 3072 ≈ 4.7M per layer. Across 12 BERT layers: 56.6M — this is why FFN is the largest single component in most transformers.

In [ ]:
def relu(x):
    """ReLU activation: max(0, x)"""
    return np.maximum(0, x)


def gelu(x):
    """GELU activation (used in modern transformers like GPT)."""
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


class FeedForward:
    """Feed-Forward Network: expand → activate → compress."""
    
    def __init__(self, d_model, d_ff, activation='relu'):
        self.W1 = np.random.randn(d_model, d_ff) * np.sqrt(2.0 / d_model)
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * np.sqrt(2.0 / d_ff)
        self.b2 = np.zeros(d_model)
        self.activation = relu if activation == 'relu' else gelu
        self.d_model = d_model
        self.d_ff = d_ff
    
    def __call__(self, x):
        # Step 1: Expand (d_model → d_ff)
        hidden = x @ self.W1 + self.b1
        
        # Step 2: Activation
        hidden = self.activation(hidden)
        
        # Step 3: Compress (d_ff → d_model)
        output = hidden @ self.W2 + self.b2
        
        return output


# Demonstrate
d_model = 8
d_ff = 32  # 4× expansion (typical: d_ff = 4 × d_model)

ffn = FeedForward(d_model, d_ff, activation='gelu')

x = np.random.randn(3, d_model)  # 3 words × 8 dims
out = ffn(x)

print(f"Feed-Forward Network:")
print(f"  Input:  {x.shape}  ({d_model} dimensions)")
print(f"  Hidden: (3, {d_ff})  ({d_ff} dimensions — 4× expansion!)")
print(f"  Output: {out.shape}  ({d_model} dimensions — back to original!)")
print(f"\n  The expansion to {d_ff} dims gives the model more 'space to think'.")
print(f"  Each word is processed INDEPENDENTLY (no interaction between words here).")

In [ ]:
# Visualize ReLU vs GELU
x_range = np.linspace(-3, 3, 200)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_range, relu(x_range), 'r-', linewidth=2.5, label='ReLU (original transformer)')
ax.plot(x_range, gelu(x_range), 'b-', linewidth=2.5, label='GELU (GPT, BERT, modern)')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)
ax.set_xlabel('Input', fontsize=13)
ax.set_ylabel('Output', fontsize=13)
ax.set_title('Activation Functions Used in Transformers', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("ReLU: sharp cutoff at 0 (simple and fast)")
print("GELU: smooth curve (slightly better performance, used in modern models)")
print("Both make the network non-linear — without them, stacking layers would be useless.")

## Part 5: The Complete Transformer Block

### Layer 1: The Assembly

You now have all the pieces:
- **Multi-Head Attention**: words talk to each other
- **Feed-Forward Network**: each word thinks privately
- **Layer Normalization**: keeps numbers stable
- **Residual Connection**: preserves original information as a safety net

One transformer block chains these together TWICE — once for attention, once for FFN:

```
Input
  │
  ├──────────────── (residual path 1)
  ▼
LayerNorm → Multi-Head Attention
  │
  + ◄──────────────── (add residual 1)
  │
  ├──────────────── (residual path 2)
  ▼
LayerNorm → Feed-Forward Network
  │
  + ◄──────────────── (add residual 2)
  │
Output (same shape as input)
```

### Layer 2: Pre-LN vs Post-LN Deep Dive

The order of LayerNorm matters significantly:

**Post-LN** (original "Attention Is All You Need"):
```
x_attn = LayerNorm(x + Attention(x))
x_out  = LayerNorm(x_attn + FFN(x_attn))
```
Problem: at initialization, x + Attention(x) has a different scale than x alone (because Attention(x) adds to it). LayerNorm after addition normalizes this, but gradients flowing back through the LayerNorm can be large early in training. This causes divergence without careful LR warmup (typically 4000 linear warmup steps).

**Pre-LN** (GPT-2, LLaMA, most modern models):
```
x_attn = x + Attention(LayerNorm(x))
x_out  = x_attn + FFN(LayerNorm(x_attn))
```
The residual stream x flows UNMODIFIED through the + operations. The gradient of loss w.r.t. x is always: ∂L/∂x_out + ∂L/∂x (from the residual). At initialization, sublayer outputs are near zero, so gradients flow mainly through the "+1" residual path. Training is stable from step 1, without warmup.

**Empirical finding**: Pre-LN is 10–30% faster to train (no warmup needed), slightly lower final quality on some benchmarks, but dramatically more stable for very deep models (48+ layers). All modern open-source LLMs use Pre-LN.

The code below implements **Pre-LN** (the modern standard).

In [ ]:
class MultiHeadAttention:
    """Multi-Head Attention (from notebook 02)."""
    
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        scale = np.sqrt(2.0 / d_model)
        self.W_Q = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_K = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_V = [np.random.randn(d_model, self.d_k) * scale for _ in range(n_heads)]
        self.W_O = np.random.randn(d_model, d_model) * scale
    
    def __call__(self, x, mask=None):
        head_outputs = []
        all_weights = []
        
        for h in range(self.n_heads):
            Q = x @ self.W_Q[h]
            K = x @ self.W_K[h]
            V = x @ self.W_V[h]
            out, w = scaled_dot_product_attention(Q, K, V, mask)
            head_outputs.append(out)
            all_weights.append(w)
        
        concatenated = np.concatenate(head_outputs, axis=-1)
        return concatenated @ self.W_O, all_weights


class TransformerBlock:
    """
    One complete transformer block (Pre-LayerNorm variant).
    
    Architecture:
        x → LayerNorm → Multi-Head Attention → + residual
          → LayerNorm → Feed-Forward          → + residual
    """
    
    def __init__(self, d_model, n_heads, d_ff):
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff, activation='gelu')
        self.ln1 = LayerNorm(d_model)
        self.ln2 = LayerNorm(d_model)
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_ff = d_ff
    
    def __call__(self, x, mask=None):
        # Sub-layer 1: Multi-Head Attention with residual connection
        x_norm = self.ln1(x)                     # Layer norm
        attn_out, attn_weights = self.attention(x_norm, mask)  # Attention
        x = x + attn_out                          # Residual connection
        
        # Sub-layer 2: Feed-Forward with residual connection
        x_norm = self.ln2(x)                     # Layer norm
        ffn_out = self.ffn(x_norm)               # Feed-forward
        x = x + ffn_out                           # Residual connection
        
        return x, attn_weights


# Test a single transformer block
d_model = 16
n_heads = 4
d_ff = 64  # 4× expansion

block = TransformerBlock(d_model, n_heads, d_ff)

# Input: 5 words, 16 dimensions each
sentence = ["The", "cat", "sat", "on", "mat"]
x = np.random.randn(len(sentence), d_model)

output, attn_weights = block(x)

print(f"Transformer Block Configuration:")
print(f"  d_model (embedding size): {d_model}")
print(f"  n_heads:                  {n_heads}")
print(f"  d_k (per head):           {d_model // n_heads}")
print(f"  d_ff (FFN inner dim):     {d_ff}")
print(f"\nInput shape:  {x.shape}")
print(f"Output shape: {output.shape}  ← same as input!")
print(f"\nThis means blocks can be STACKED — the output of one becomes")
print(f"the input of the next. Transformers typically stack 6-96 blocks.")

## Part 6: Stacking Multiple Blocks

Real transformers stack many blocks. Each block refines the representation:
- **Early blocks**: simple patterns (grammar, nearby words)
- **Middle blocks**: combining information
- **Later blocks**: complex patterns (meaning, reasoning)

In [ ]:
class MiniTransformer:
    """A complete (tiny) transformer: embeddings + positional encoding + N blocks."""
    
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len=100):
        self.d_model = d_model
        self.n_layers = n_layers
        
        # Embedding table (word → vector)
        self.embedding = np.random.randn(vocab_size, d_model) * 0.02
        
        # Positional encoding
        self.pos_encoding = sinusoidal_positional_encoding(max_len, d_model)
        
        # Stack of transformer blocks
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        
        # Final layer norm
        self.final_ln = LayerNorm(d_model)
    
    def forward(self, token_ids, mask=None):
        seq_len = len(token_ids)
        
        # Step 1: Token embeddings
        x = self.embedding[token_ids]  # Look up each token
        
        # Step 2: Add positional encoding
        x = x + self.pos_encoding[:seq_len]
        
        # Step 3: Pass through each transformer block
        all_attention = []
        for block in self.blocks:
            x, attn = block(x, mask)
            all_attention.append(attn)
        
        # Step 4: Final layer norm
        x = self.final_ln(x)
        
        return x, all_attention


# Create a mini transformer
vocab_size = 100
d_model = 32
n_heads = 4
d_ff = 128
n_layers = 4

transformer = MiniTransformer(vocab_size, d_model, n_heads, d_ff, n_layers)

# Simulate processing a sentence
sentence = ["The", "cat", "sat", "on", "the", "mat"]
token_ids = np.array([10, 42, 67, 23, 10, 55])  # Fake token IDs

output, all_attention = transformer.forward(token_ids)

print(f"Mini Transformer Configuration:")
print(f"  Vocabulary size:  {vocab_size}")
print(f"  d_model:          {d_model}")
print(f"  n_heads:          {n_heads}")
print(f"  d_ff:             {d_ff}")
print(f"  n_layers:         {n_layers}")
print(f"")
print(f"Input:  {len(token_ids)} tokens")
print(f"Output: {output.shape}  ({len(token_ids)} words × {d_model} dims)")
print(f"")
print(f"Total parameter count (approximate):")

# Count parameters
n_params = vocab_size * d_model  # Embeddings
per_block = (3 * n_heads * d_model * (d_model // n_heads)  # Q, K, V projections
             + d_model * d_model   # W_O
             + d_model * d_ff + d_ff  # FFN layer 1
             + d_ff * d_model + d_model  # FFN layer 2
             + 2 * 2 * d_model)   # 2 LayerNorms
n_params += per_block * n_layers
print(f"  ~{n_params:,} parameters")
print(f"  (Real GPT-3 has 175,000,000,000 — 175 billion!)")

## Part 7: Visualize Attention Across Layers

Let's see how attention patterns evolve through the layers:

In [ ]:
# Visualize attention from all layers (showing head 0 from each layer)
fig, axes = plt.subplots(1, n_layers, figsize=(18, 4.5))

for layer_idx in range(n_layers):
    ax = axes[layer_idx]
    # Show head 0 from each layer
    w = all_attention[layer_idx][0]  # Layer's head 0
    
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=0.8)
    ax.set_xticks(range(len(sentence)))
    ax.set_yticks(range(len(sentence)))
    ax.set_xticklabels(sentence, fontsize=9, rotation=45, ha='right')
    ax.set_yticklabels(sentence, fontsize=9)
    ax.set_title(f'Layer {layer_idx + 1}\n(Head 1)', fontsize=12)
    
    if layer_idx == 0:
        ax.set_ylabel('Query (FROM)', fontsize=11)

fig.suptitle('Attention Patterns Across Layers (Head 1 of each layer)',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("In a trained model, you would see:")
print("  Early layers: local patterns (nearby words, punctuation)")
print("  Middle layers: syntactic patterns (subject-verb, modifier-noun)")
print("  Later layers: semantic patterns (meaning, coreference)")
print("\nOur model is randomly initialized, so patterns are not meaningful yet.")
print("After training, the patterns become interpretable!")

## Part 8: The Complete Architecture Diagram

Let's trace the full data flow and verify shapes at each step:

In [ ]:
print("""Complete Transformer Data Flow
================================

Input: "The cat sat on the mat" (6 tokens)
""")

# Trace through each step
token_ids = np.array([10, 42, 67, 23, 10, 55])
seq_len = len(token_ids)

# Step 1: Embedding
x = transformer.embedding[token_ids]
print(f"Step 1: Token Embedding")
print(f"  token_ids → embedding table lookup")
print(f"  Shape: {token_ids.shape} → {x.shape}  (6 tokens × 32 dims)")

# Step 2: Positional encoding
pe = transformer.pos_encoding[:seq_len]
x = x + pe
print(f"\nStep 2: + Positional Encoding")
print(f"  x = word_embedding + position_encoding")
print(f"  Shape: {x.shape}  (unchanged — position is ADDED)")

# Step 3: Transformer blocks
for i, block in enumerate(transformer.blocks):
    x_before = x.copy()
    
    # Sub-layer 1: Attention
    x_norm = block.ln1(x)
    attn_out, _ = block.attention(x_norm)
    x = x + attn_out
    
    # Sub-layer 2: FFN
    x_norm = block.ln2(x)
    ffn_out = block.ffn(x_norm)
    x = x + ffn_out
    
    print(f"\nStep 3.{i+1}: Transformer Block {i+1}")
    print(f"  LayerNorm → Multi-Head Attn → + residual → LayerNorm → FFN → + residual")
    print(f"  Shape: {x.shape}  (always {seq_len} × {d_model})")

# Step 4: Final layer norm
x = transformer.final_ln(x)
print(f"\nStep 4: Final Layer Norm")
print(f"  Shape: {x.shape}")

print(f"\n{'=' * 50}")
print(f"Final output: {x.shape}")
print(f"Each word now has a rich, context-aware representation.")
print(f"\nTo make predictions (next word, classification, etc.),")
print(f"you'd add a task-specific head on top:")
print(f"  - Language model: Linear(d_model → vocab_size) + softmax")
print(f"  - Classifier: Linear(d_model → num_classes) + softmax")

## Part 9: Parameter Count Breakdown

Let's understand where all the parameters live in a transformer:

In [ ]:
configs = [
    ("Our Mini Model", 100, 32, 4, 128, 4),
    ("GPT-2 Small", 50257, 768, 12, 3072, 12),
    ("BERT Base", 30522, 768, 12, 3072, 12),
    ("GPT-3", 50257, 12288, 96, 49152, 96),
]

print(f"{'Model':<16} {'Vocab':>8} {'d_model':>8} {'Heads':>6} {'d_ff':>6} "
      f"{'Layers':>7} {'Total Params':>15}")
print("─" * 75)

for name, vocab, dm, heads, dff, layers in configs:
    # Embeddings
    emb_params = vocab * dm
    
    # Per block
    attn_params = 4 * dm * dm  # W_Q, W_K, W_V, W_O (simplified)
    ffn_params = 2 * dm * dff  # Two linear layers
    ln_params = 4 * dm         # 2 layer norms × 2 params each
    block_params = attn_params + ffn_params + ln_params
    
    total = emb_params + block_params * layers
    
    if total >= 1e9:
        total_str = f"{total/1e9:.1f}B"
    elif total >= 1e6:
        total_str = f"{total/1e6:.1f}M"
    else:
        total_str = f"{total/1e3:.1f}K"
    
    print(f"{name:<16} {vocab:>8,} {dm:>8} {heads:>6} {dff:>6} "
          f"{layers:>7} {total_str:>15}")

print("\nWhere do the parameters live?")
print("  ~30% in attention (W_Q, W_K, W_V, W_O)")
print("  ~65% in feed-forward networks (two large linear layers)")
print("  ~5%  in embeddings and layer norms")
print("\nThe FFN is actually the biggest component by parameter count!")

## Summary

We built a complete transformer from scratch! Here's everything we assembled:

```
Token IDs → Embedding + Positional Encoding → [Transformer Block × N] → Output

Each Transformer Block:
  ┌─ LayerNorm → Multi-Head Attention → + residual ─┐
  └─ LayerNorm → Feed-Forward Network → + residual ─┘
```

### Components Recap

| Component | Purpose | Analogy |
|-----------|---------|--------|
| Embedding | Words → numbers | Dictionary lookup |
| Positional Encoding | Add position info | Seat numbers |
| Multi-Head Attention | Words talk to each other | Group discussion |
| Feed-Forward Network | Each word thinks privately | Quiet reflection |
| Layer Normalization | Keep numbers stable | Grading on a curve |
| Residual Connection | Preserve original signal | Keeping a backup |

### Key Takeaways

1. A **transformer block** = attention + FFN + layer norm + residuals
2. Blocks can be **stacked** because output shape = input shape
3. **Residual connections** prevent information loss in deep networks
4. **Layer normalization** keeps training stable
5. **FFN** is where most parameters live (~65%)
6. The whole architecture is **parallelizable** — that's why GPUs love transformers

### What's Next?

With this foundation, you're ready to:
- Implement a transformer in PyTorch
- Understand how GPT, BERT, and other models are structured
- Move on to [fine-tuning](../../02-fine-tuning/) to adapt pre-trained transformers

## Layer 2: Expert Depth

### Full Parameter Count Breakdown (Per Block)

For one transformer block with d_model = 768, n_heads = 12, d_ff = 3072:

| Component | Computation | Parameters |
|-----------|------------|------------|
| Q projection | 3 heads × d_model × d_k | 3 × 768² = 1,769,472 |
| W_O projection | d_model × d_model | 768² = 589,824 |
| Attention total | | **4 × d_model² = 2,359,296** |
| FFN W₁ | d_model × d_ff | 768 × 3072 = 2,359,296 |
| FFN W₂ | d_ff × d_model | 3072 × 768 = 2,359,296 |
| FFN biases | d_ff + d_model | ~4,000 (negligible) |
| FFN total | | **2 × d_model × d_ff ≈ 4,718,592** |
| 2 × LayerNorm | 2 × (γ + β) | 2 × 2 × 768 = 3,072 |
| **Block total** | | **~7,081,000** |

BERT Base: 12 blocks × 7.08M = 85M (plus embeddings ~24M = 109M total ✓)

Key insight: **FFN has ~2× more parameters than attention**. The "attention is the heart of transformers" narrative is true for computation (n² scaling), but FFN holds most of the weights.

### Pre-LN Gradient Flow Analysis

At initialization with Pre-LN, the variance of the residual stream stays controlled:

For L stacked Pre-LN blocks: Var[x_L] = Var[x_0] × (1 + O(1/d_model)) per block. After L blocks: Var[x_L] ≈ Var[x_0] × (1 + ε)^L ≈ Var[x_0] for small ε.

With Post-LN: LayerNorm forces Var = 1 at each step, but the GRADIENT of LayerNorm is ∂LN/∂x = (I - ¹/d 1 1^T) / σ. For large gradients flowing back, σ is small early in training (because activations are small), so 1/σ is large → large gradient magnitudes → instability without warmup.

### The FFN as Key-Value Memory

Geva et al. 2021 showed each FFN hidden unit stores a key-value pair:
- **Key** = W₁[i] (the i-th row of the first weight matrix)
- **Value** = W₂[:, i] (the i-th column of the second weight matrix)
- **Activation** = ReLU(W₁ x)_i — fires when input x aligns with key W₁[i]

Practically: you can find which FFN neurons activate for "Python programming" and suppress the corresponding values to make the model "forget" Python. This is the basis of model editing techniques (ROME, MEMIT) that surgically update factual knowledge without full fine-tuning.

### Staff/Principal Interview Q&A

**Q: Why is Pre-LN more stable than Post-LN? What's the gradient flow argument?**

In Post-LN, after each block: x_out = LN(x + sublayer(x)). The gradient flowing to x is: ∂L/∂x = ∂L/∂x_out × ∂LN/∂(x+s) × (1 + ∂s/∂x). The LN Jacobian ∂LN/∂z ≈ I/σ(z), where σ(z) is the std of z. Early in training with small activations, σ is small → ∂LN is large → gradient magnitude spikes → divergence. In Pre-LN: x_out = x + sublayer(LN(x)). Gradient to x: ∂L/∂x = ∂L/∂x_out × (1 + ∂s/∂x × ∂LN/∂x). The "+1" from the residual path provides a constant gradient floor. The LN Jacobian multiplies only the sublayer path, not the residual path. So even if LN creates large gradients in the sublayer path, the residual path gradient is unaffected, preventing explosion.

**Q: Describe two specific ways to reduce transformer inference cost, beyond quantization.**

Speculative decoding: run a small "draft model" to generate k token candidates, then verify them all in parallel with the large model. If the large model accepts them, you generate k tokens in one large-model pass instead of k passes. Speedup is 2–4× for typical token acceptance rates, at no quality cost (it's exact). KV cache compression: standard KV cache grows linearly with context length. Techniques: (1) Sliding window attention (Mistral) — each token attends only to the last W tokens. KV cache capped at W × n_layers × 2 × d_k. Quality tradeoff: distant tokens are invisible. (2) Multi-query attention — all query heads share one K,V projection: cache shrinks from n_heads × to 1×. (3) Key-value quantization — store K,V in int8 instead of float16: 2× memory reduction with <1% quality loss on most benchmarks.

**Q: What would change in the architecture if you doubled d_ff from 4× to 8× d_model?**

Parameters: FFN grows from 2 × d_model × 4d_model = 8d_model² to 2 × d_model × 8d_model = 16d_model². This doubles FFN parameters (which are already 2× attention parameters). Total model would be ~1.4× larger. Training FLOPs: also roughly doubles (since FFN is the main contributor for moderate n). Typical empirical result: +1–3% on downstream benchmarks, diminishing returns beyond 4×. The 4× ratio is a well-calibrated sweet spot — tried in original paper and repeatedly validated. Notable exception: Mixture-of-Experts (MoE) models like Mixtral effectively use much larger d_ff but route each token to only 2 of 8 experts, achieving 8× d_ff at 2× compute per token. This separates the "parameter count → capacity" benefit from the "d_ff × d_model → FLOPs" cost.

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)